In [1]:
import torch
import shap
import re
import numpy as np
import os
import wandb
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from huggingface_hub import login
from weave.scorers import HallucinationFreeScorer

# ------ ollama config ---------
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL_NAME = "llama3.1"

# ------- hugging face config ---------
login(token="hf_token")

# ------ model and tokenizer config --------
model_name = "Meta-Llama-3.1-8B-Instruct"
model_path = os.path.abspath("/Users/allison/Documents/Models/"+model_name)
tokenizer = AutoTokenizer.from_pretrained(model_path)

# ------ stringDB -------
SPECIES = [9606, 10090, 4932, 7227] # human, mouse, yeast, fruit fly

/Users/allison/anaconda3/envs/protein-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading checkpoint shards: 100%|██████████| 4/4 [00:09<00:00,  2.26s/it]


In [3]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    temperature=0.4,
    do_sample=False,
    return_full_text=False
)

Device set to use mps
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [ ]:
import requests
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

def create_system_prompt():
    """
    Create the system prompt for protein interaction queries.
    This could be enhanced with additional context or instructions.
    """
    return "You are a helpful assistant specialized in molecular biology and protein interactions. When asked about protein interactions, provide clear, concise lists of interacting proteins with brief explanations. Focus on generating accurate protein names that are commonly used in databases and literature."

def create_user_prompt(protein_name):
    """
    Create the user prompt for a specific protein.
    
    Args:
        protein_name (str): The name of the protein to query about
        
    Returns:
        str: The formatted user prompt
    """
    base_prompt = (
        "List proteins that might interact with {protein}. "
        "Please provide a simple list of protein names (gene symbols/names) "
        "that could potentially interact with {protein}, along with a brief "
        "reason for each interaction. Be specific and accurate in your reasoning."
        "Focus on well-known, documented interactions. "
        "For each interaction, write the interacting protein first, then a dash '-', "
        "then a brief description of the interaction type."
        "Do not repeat {protein} as the interacting protein."
        "Format your response as: INTERACTING_PROTEIN_NAME - brief description of interaction type."
    )
    return base_prompt.format(protein=protein_name)

def parse_ollama_output(text):
    """
    Extracts protein names and descriptions from LLM ouptut.
    """
    items = re.findall(r"\d+\.\s+\*\*(.*?)\*\*\s*-\s*(.*)", text)
    protein_list = [{"name": name.strip(), "description": desc.strip()} for name, desc in items]
    return protein_list

def normalize_protein_name(name):
    """Normalize protein/gene names: remove non-alphanumeric, uppercase, apply aliases."""
    CHAR_MAP = {
        'α': 'A', 'β': 'B', 'γ': 'G', 'δ': 'D', 'ε': 'E', 'κ': 'K', 'λ': 'L',  'μ': 'M', 'ν': 'N', 'ρ': 'R', 'σ': 'S', 'τ': 'T', 'φ': 'F', 'χ': 'X', 'ψ': 'Y', 'ω': 'W',
    }
    name = name.strip()
    for char, letter in CHAR_MAP.items():
        name = name.replace(char, letter)
    name = re.sub(r'[^A-Za-z0-9]', '', name)
    return name.upper()

def parse_llm_output(text, source_name=None):
    """
    Parse LLM output into a list of {"name": target_name, "description": desc} dicts.
    Handles:
        - single-dash lines: TARGET - description
        - double-dash lines: SOURCE - TARGET - description
        - markdown (**protein**)
        - multi-line descriptions
        - normalization of protein names

    Args:
        text (str): LLM output
        source_name (str, optional): Source protein name, for double-dash lines

    Returns:
        list of dict: [{"name": target_name, "description": description}, ...]
    """
    parsed_interactions = []
    lines = text.split('\n')
    
    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Remove markdown formatting
        line = re.sub(r'\*\*([^*]+)\*\*', r'\1', line)
        # Remove numbering / bullets
        line = re.sub(r'^\s*[\d\.\)\-•]+\s*', '', line)
        dash_count = line.count(' - ')

        if dash_count == 0:
            continue
        elif dash_count == 1:
            # TARGET - description
            target, description = line.split(' - ', 1)
        else:
            # SOURCE - TARGET - description
            parts = line.split(' - ', 2)
            if source_name and parts[0].upper() == source_name.upper():
                target, description = parts[1], parts[2] if len(parts) > 2 else ''
            else:
                target, description = parts[1], parts[2] if len(parts) > 2 else parts[1]

        target = normalize_protein_name(target)
        description = description.strip()

        parsed_interactions.append({
            "name": target,
            "description": description
        })

    return parsed_interactions


def llm_predict(protein_names):
    prompts = [create_system_prompt() + "\n\n" + create_user_prompt(p) for p in protein_names]
    results = pipe(prompts) 
    texts = [r[0]["generated_text"] for r in results]
    return texts

def get_string_partners(source, source_id, target, target_id, species):
    """
    Fetching predicted functional partners from STRINGDB
    
    Args: 
        protein_name (str): Gene symbol of protein (e.g., 'PTEN')
        species (int): NCBI taxonomy ID (e.g., 9606)
    
    Returns:
        list of dict: each dict contains partner name, description, score
    """
    url = f"https://string-db.org/api/json/network?identifiers={source_id}%0d{target_id}&species={species}"
    response = requests.get(url)
    response.raise_for_status() 
    data = response.json()
    scores = {}
    score_types = ['score', 'nscore', 'fscore', 'pscore', 'ascore', 'escore', 'dscore', 'tscore']
    if (len(data)): # interaction found
        entry = data[0]
        scores = {field: entry.get(field, 0.0) for field in score_types}
        if (
            (str(entry.get('stringId_A')).lower() == str(source_id).lower() \
                and str(entry.get('stringId_B')).lower() == str(target_id).lower()) or \
            (str(entry.get('stringId_A')).lower() == str(target_id).lower() \
                and str(entry.get('stringId_B')).lower() == str(source_id).lower())
        ):
            scores["source"] = str(entry.get('preferredName_A')).upper()
            scores["target"] = str(entry.get('preferredName_B')).upper()
            scores["ncbiTaxonId"] = species 
            scores["interaction"] = True 
            return scores 
        
    # no interaction found 
    scores = {field: 0.0 for field in score_types}
    scores["source"] = source 
    scores["target"] = target 
    scores["ncbiTaxonId"] = species 
    scores["interaction"] = False 
    return scores

def get_string_info(protein, species):
    url = f"https://string-db.org/api/json/get_string_ids"
    params = {
        "identifiers": protein,
        "species": species
    }
    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        if data:
            return {
                "name": data[0].get("preferredName"),
                "stringId": data[0].get("stringId"),
                "ncbiTaxonId": data[0].get("ncbiTaxonId"),
                "annotation": data[0].get("annotation", "")
            }
    except Exception as e:
        print(f"Error resolving {protein} in species {species}: {e}")
    return None

def description_similarity(textA, textB):
    if not textA or not textB:
        return 0
    emb1 = embed_model.encode(textA, convert_to_tensor=True)
    emb2 = embed_model.encode(textB, convert_to_tensor=True)
    return float(util.pytorch_cos_sim(emb1, emb2))

def process_interaction(source, target, llm_reasoning):
    """
    Checks if two proteins occur in the same species. 
    If so, checks for a known interaction. 
    If proteins do not occur in the same species or 
    occur in the same species but do not have a known interaction, 
    computes description similarity. 

    Args: 
        source (str): Gene symbol of given protein
        target (str): Gene symbol of predicted interacting protein
        llm_reasoning (str): short description of interaction given by LLM
    
    Returns:
        list of dict: interactions across species with interaction/hallucination scores and reasoning. 
    """
    interaction = {
        "source": source, 
        "target": target, 
        "reasoning": llm_reasoning,
        "overall_score": 0.0,
        "interacting_species": [],
        "shared_species": [],
        "scores": [],
    }
    feasibility_scores = []
    known_species = []
    shared_species = []
    shared = False

    for species in SPECIES:
        src_info = get_string_info(source, species)
        tgt_info = get_string_info(target, species)

        if src_info and tgt_info: # shared species
            shared = True
            shared_species.append(species)
            interaction["shared_species"] = shared_species
            src_annotation = src_info["annotation"]
            tgt_annotation = tgt_info["annotation"]

            # checking for known interaction
            interaction_data = get_string_partners(source, src_info["stringId"], target, tgt_info["stringId"], species)
            if interaction_data["interaction"]: # interaction found
                known_species.append(species)
                interaction["interacting_species"] = known_species
                feasibility_scores.append({
                    **interaction_data
                })
            else: 
                sim = description_similarity(src_annotation, tgt_annotation)
                feasibility_scores.append({
                    "source": str(src_info["name"]).upper(),
                    "target": str(tgt_info["name"]).upper(),
                    "src_annotation": src_annotation,
                    "tgt_annotation": tgt_annotation,
                    "score": -round(sim, 4),
                    "ncbiTaxonId": species,
                    "interaction": False
                    # **interaction_data
                })
        
    if not shared: # proteins do not share a species--target protein not found
        interaction["overall_score"] = -1.0
        return interaction
        
    interaction["scores"] = feasibility_scores 
    interaction["overall_score"] = max([f["score"] for f in feasibility_scores])

    return interaction


def generate_interaction_scores(results, part=1):
    """
    Predicts interacting proteins with those in the given list and a description of the interaction.
    Processes the interactions by giving them feasbility scores based on whether the interaction exists
    and the similarity of the given description to the functional annotation of the predicted protein.
    
    Args: 
        list of str: list of proteins to generate predicted interactions from.
    
    Returns:
        list of dict: protein names and info for each predicted interaction. 
    """
    scores = [] 
    
    with ThreadPoolExecutor(max_workers=8) as executor:
        futures = []
        for pr in results:
            node = pr["node"]
            for pred in pr["links"]:
                futures.append(
                    executor.submit(
                        process_interaction,
                        node,  # protein A
                        pred["name"],  # protein B
                        pred["description"]  # description text
                    )
                )

        for future in as_completed(futures):
            try:
                result = future.result()
                scores.append(result)
            except Exception as e:
                print(f"Error in processing: {e}")
    
    with open(f"graph_master_part100_{part}.json", "w") as f:
        json.dump(scores, f, indent=2)

### List of proteins for prediction

In [234]:
protein_names = ['AAK1'] # ['PTEN', 'TRIM3']
results = llm_predict(protein_names)
parsed_results = [
    {"node": protein_names[i], 
        "links": [parse_llm_output(results[i])][0]} for i in range(len(protein_names))
    ]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [235]:
scores = generate_interaction_scores(parsed_results)

### PPI Prediction using RAG
https://medium.com/data-science/visualize-your-rag-data-evaluate-your-retrieval-augmented-generation-system-with-ragas-fc2486308557

In [50]:
from langchain_core.documents import Document
from sklearn.metrics import precision_recall_fscore_support
import csv
from io import StringIO
import requests
import hashlib

def create_system_prompt():
    """
    Create the system prompt for protein interaction queries.
    This could be enhanced with additional context or instructions.
    """
    return "You are a helpful assistant specialized in molecular biology and protein interactions. When asked about protein interactions, provide clear, concise lists of interacting proteins with brief explanations. Focus on generating accurate protein names that are commonly used in databases and literature."

def create_user_prompt(protein):
    """
    Build the LLM prompt with context from retrieval.
    """
    prompt = f"""
    {create_system_prompt()}

    Predict proteins that interact with {protein}, 
    providing the interacting protein name and a short reasoning for each.
    Use the following format for each prediction: PROTEIN_NAME - reasoning
    """
    return prompt.strip()

def stable_hash_meta(metadata: dict) -> str:
    """
    Create a stable hash string for a metadata dictionary.
    Ensures consistent ordering of keys so that the same
    metadata always gives the same hash.
    """
    # Dump the dict to a JSON string with sorted keys
    meta_str = json.dumps(metadata, sort_keys=True)
    # Return a short stable hash
    return hashlib.md5(meta_str.encode("utf-8")).hexdigest()

def get_protein_info(protein_name):
    """
    Retrieve context for a protein from UniProt.
    """
    base_url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": {protein_name},
        "format": "tsv",
        "fields": (
            "accession,id,protein_name,gene_names,organism_name,"
            "go_f,go_c,go_p,"   # Gene Ontology terms (molecular function, cellular component, biological process)
            "xref_reactome,xref_kegg,keyword"  # pathways and keywords
        )
    }

    try:
        r = requests.get(base_url, params=params, timeout=10)
        r.raise_for_status()
        return r.text
    except Exception as e:
        print(f"[Retriever Error] {e}")
        return ""

def parse_rag_output(text):
    """
    Parse LLM/RAG output into a list of dicts with 'name' and 'description'.
    Handles numbered lists, repeated lines, and extra whitespace.
    """
    interactions = []
    seen = set()
    
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        
        line = re.sub(r'^[\d\.\-\•\s>]+', '', line)
        if '-' not in line:
            continue
        
        name, desc = line.split('-', 1)
        name = name.strip()
        desc = desc.strip()
        
        if name in seen:
            continue
        seen.add(name)
        
        interactions.append({"name": name, "description": desc})
    
    return interactions

def parse_context(raw_text):
    """
    Parse raw context into a structured list of dicts.
    Each line is expected to be tab-separated.
    """
    f = StringIO(raw_text)
    reader = csv.DictReader(f, delimiter="\t")
    contexts = []
    for row in reader:
        # collecting GO terms
        go_terms = []
        for go_field in [
            "Gene Ontology (biological process)",
            "Gene Ontology (molecular function)",
            "Gene Ontology (cellular component)"
        ]:
            if row.get(go_field):
                go_terms.extend([term.strip() for term in row[go_field].split(";") if term.strip()])

        # collecting pathways
        pathways = []
        for pw_field in ["Reactome", "KEGG"]:
            if row.get(pw_field):
                pathways.extend([pw.strip() for pw in row[pw_field].split(";") if pw.strip()])

        # collecting gene sets
        gene_sets = []
        if row.get("Keywords"):
            gene_sets = [kw.strip() for kw in row["Keywords"].split(";") if kw.strip()]

        contexts.append({
            "accession": row.get("Entry", ""),
            "id": stable_hash_meta(row.get("Entry Name", "")),
            "name": row.get("Entry Name", ""),
            "description": row.get("Protein names", ""),
            "synonyms": row.get("Gene Names", ""),
            "organism": row.get("Organism", ""),
            "go_terms": go_terms,
            "pathways": pathways,
            "gene_sets": gene_sets,
        })

    return contexts

def format_context(parsed_contexts):
    """
    Turn parsed dicts into a readable string for the LLM.
    """
    formatted_lines = []
    for item in parsed_contexts:
        formatted_lines.append(
            f"- {item['id']} ({item['organism']}): {item['description']}"
        )
    return "\n".join(formatted_lines)

def fetch_known_interactors(protein_id, species=9606, limit=50):
    """
    Fetch known interactors from STRING for a given protein.
    protein_id: UniProt/STRING identifier (e.g., "PEX10")
    species: NCBI Taxon ID (9606 for human)
    limit: max number of interactors
    """
    url = f"https://string-db.org/api/json/interaction_partners?identifiers={protein_id}&species={species}"
    try:
        r = requests.get(url)
        r.raise_for_status()
        data = r.json()
        interactors = []
        for edge in data:
            partner = edge.get("preferredName_B")
            if partner:
                interactors.append(partner)
        return set(interactors)
    except Exception as e:
        print(f"Error fetching interactors for {protein_id}: {e}")
        return set()

# def parse_predictions(answer_text):
#     """
#     Extract predicted protein partners from LLM output.
#     Accepts protein names with letters, numbers, underscores, hyphens.
#     """
#     preds = set()
#     desc = set()
#     for line in answer_text.splitlines():
#         line = line.strip()
#         if not line or "-" not in line:
#             continue
#         # Remove leading bullets, numbers, dots, spaces
#         line = re.sub(r"^[\d\.\-\•\s>]+", "", line)
#         # Take everything before the first dash as the protein name and everything after as the reasoning
#         protein_name = line.split("-", 1)[0].strip()
#         protein_desc = line.split("-", 1)[1].strip() if "-" in line else ""
#         if protein_name:
#             preds.add(protein_name)
#             desc.add(protein_desc)
#     return list(preds), list(desc)

def parse_predictions(raw_answer):
    """
    Extract protein predictions + reasoning from LLM answer text.
    Handles both numbered lists and inline "PROTEIN - description" formats.
    """

    preds, reasoning = [], []

    numbered_lines = re.findall(r'^\s*\d+\.\s*(.*?)$', raw_answer, re.MULTILINE)
    if numbered_lines:
        for line in numbered_lines:
            if " - " in line:
                prot, desc = line.split(" - ", 1)
            else:
                parts = line.split(maxsplit=1)
                prot, desc = parts[0], parts[1] if len(parts) > 1 else ""
            prot = prot.strip()
            desc = desc.strip()

            if re.match(r'^[A-Z0-9-]{2,}$', prot):
                preds.append(prot)
                reasoning.append(desc)

    if not preds:
        matches = re.findall(r'\b([A-Z0-9-]{2,})\s*-\s*([^.\n]+)', raw_answer)
        for prot, desc in matches:
            prot = prot.strip()
            desc = desc.strip()
            if prot not in preds:
                preds.append(prot)
                reasoning.append(desc)

    bad_tokens = {"Note", "Helpful", "Answer"}
    cleaned = [(p, r) for p, r in zip(preds, reasoning) if p not in bad_tokens]

    preds, reasoning = zip(*cleaned) if cleaned else ([], [])
    return list(preds), list(reasoning)


def evaluate_predictions(preds, ground_truth):
    y_true = [1 if p in ground_truth else 0 for p in preds]
    y_pred = [1] * len(preds)  # all LLM outputs are predictions
    if not ground_truth:
        return {"precision": 0, "recall": 0, "f1": 0}
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    return {"precision": precision, "recall": recall, "f1": f1}

def create_docs_from_context(contexts):
    docs = []
    for c in contexts:
        go_terms = c.get("go_terms", [])
        pathways = c.get("pathways", [])
        gene_sets = c.get("gene_sets", [])

        page_content = (
            f"{c['id']}: {c['description']}. "
            f"Synonyms: {c['synonyms']}. "
            f"Organism: {c['organism']}. "
            f"GO Terms: {', '.join(go_terms)}. "
            f"Pathways: {', '.join(pathways)}. "
            f"Gene Sets: {', '.join(gene_sets)}."
        )
        metadata = {
            **c,
            "go_terms": go_terms,
            "pathways": pathways,
            "gene_sets": gene_sets,
        }

        docs.append(
            Document(
                page_content=page_content,
                metadata=metadata
            )
        )
        
    return docs 

def llm_rag_predict(protein):
    """
    Simple RAG predictor
    """
    context = get_protein_info(protein)
    parsed_contexts = parse_context(context)
    formatted_context = format_context(parsed_contexts)
    rag_prompt = build_rag_prompt(protein, formatted_context)
    
    results = pipe([{"role": "user", "content": rag_prompt}])
    return results[0]["generated_text"], parsed_contexts

In [5]:
# from langchain.vectorstores import FAISS
import faiss
import json
import os
from langchain.docstore import InMemoryDocstore

def save_faiss_index(vectorstore, folder_path: str):
    os.makedirs(folder_path, exist_ok=True)
    faiss.write_index(vectorstore.index, os.path.join(folder_path, "index.faiss"))
    docs_out = []
    for doc in vectorstore.docstore._dict.values():
        docs_out.append({"page_content": doc.page_content, "metadata": doc.metadata})
    with open(os.path.join(folder_path, "docs.json"), "w") as f:
        json.dump(docs_out, f)

def load_faiss_index(folder_path: str, embedding_model):
    with open(folder_path + "/docs.json", "rb") as f:
        data = json.load(f)

    docs = [Document(page_content=d["page_content"], metadata=d["metadata"]) for d in data]
    index = faiss.read_index(folder_path + "/index.faiss")
    index_to_docstore_id = {i: str(i) for i in range(len(docs))}
    vectorstore = FAISS(
        embedding_function=embedding_model,
        index=index,
        docstore=InMemoryDocstore({str(i): doc for i, doc in enumerate(docs)}),
        index_to_docstore_id=index_to_docstore_id
    )

    return vectorstore

Saving prompts and documents/context for predictions

Creating vectorstore for mapping contexts. ```rag_chain``` passes query to the chain, and the retriever finds the top k (5) documents from the vectorstore and passes those to the LLM as context in the prompt.

In [55]:
from langchain.llms import HuggingFacePipeline
from langchain.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer, util
from langchain.chains import RetrievalQA
from langchain_community.vectorstores.faiss import FAISS
import pandas as pd


# ---- model setup ------
llm = HuggingFacePipeline(pipeline=pipe)
# embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2") # "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
all_docs = []
protein_queries = ["AAK1", "PTEN", "KISS1", "TP53"]

# building vectorstore for RAG
for protein in protein_queries:
    context = get_protein_info(protein)
    parsed_contexts = parse_context(context)
    docs = create_docs_from_context(parsed_contexts)
    all_docs.extend(docs)

vectorstore_path = "protein_vectorstore"
vectorstore = FAISS.from_documents(all_docs, embedding_model)
save_faiss_index(vectorstore, vectorstore_path)

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k":10})

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff", 
    return_source_documents=True 
)

In [45]:
def get_predictions(protein):
    """
    Main function to get predictions for a protein using RAG pipeline. 
    
    Returns: 
    - predictions
    - context used for prompt 
    - evaluation metrics
    - ground truth used in evaluation 
    - source documents used by RAG
    - raw answer from LLM
    """
    prompt = create_user_prompt(protein)
    response = rag_chain.invoke(prompt)

    # LLM output
    answer = response['result']
    preds, reasons = parse_predictions(answer)

    # text content of context/source documents 
    context_used = "\n".join([doc.page_content for doc in response["source_documents"]])
    source_docs = [
        # stable_hash_meta(doc.metadata) for doc in response["source_documents"]
        doc.metadata.get("id", "") if doc.metadata else "" for doc in response["source_documents"]
    ]

    # embeddings of retrieved documents
    embeddings = [
        embedding_model.embed_query(doc.page_content) 
        for doc in response["source_documents"]
    ]

    # evaluating predictions
    ground_truth = fetch_known_interactors(protein)
    metrics = evaluate_predictions(preds, ground_truth)

    return {
        "protein": protein,
        "predictions": preds,
        "reasoning": reasons,
        "ground_truth": list(ground_truth),
        "metrics": metrics,
        "contexts": context,
        "source_documents": source_docs,
        "embedding": embeddings[0],
        "raw_answer": answer
    }

In [56]:
results = []

for protein in protein_queries:
    res = get_predictions(protein)
    results.append(res)

df_qa_eval = pd.DataFrame(results)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


RuntimeError: MPS backend out of memory (MPS allocated: 67.93 GiB, other allocations: 13.83 GiB, max allowed: 81.60 GiB). Tried to allocate 166.28 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

Generating embeddings for questions and contexts

In [48]:
df_questions_exploded = df_qa_eval.explode("source_documents")

df_docs = pd.DataFrame([{
    "name": doc.metadata.get("name"),
    "accession": doc.metadata.get("accession"),
    "synonyms": doc.metadata.get("synonyms"),
    "organism": doc.metadata.get("organism"),
    "source_documents": doc.metadata.get("id"),  
    "description": doc.metadata.get("description"),
    "go_terms": ", ".join(doc.metadata.get("go_terms", [])),
    "pathways": ", ".join(doc.metadata.get("pathways", [])),
    "gene_sets": ", ".join(doc.metadata.get("gene_sets", [])),
} for doc in all_docs])

agg = (
    df_questions_exploded.groupby("source_documents")
    .agg(
        num_questions=("protein", "count"), # count of questions referencing the document
        question_ids=("protein", lambda x: list(x)), # list of question IDs referencing the document
    )
    .reset_index()
)
# merging aggregated information back into df_documents 
df_documents_agg = pd.merge(df_docs, agg, on="source_documents", how="left").rename(columns={"source_documents": "id"})
df_documents_agg["question_ids"] = df_documents_agg["question_ids"].apply(lambda x: x if isinstance(x, list) else [])
# replace NaN values in 'num_questions' with 0
df_documents_agg["num_questions"] = df_documents_agg["num_questions"].fillna(0).astype(int)
df = df_questions_exploded.merge(
    df_documents_agg,
    left_on="source_documents",
    right_on="id",
    how="left"
)

In [54]:
df_qa_eval

,protein,predictions,reasoning,ground_truth,metrics,contexts,source_documents,embedding,raw_answer,reasons
0,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,"[05be9863936dea49c55ce86cd5088cb6, 658b7b83337...","[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...
1,PTEN,"[PI3K, AKT, TSC2, LKB1, PDK1, 14-3-3]",[PTEN is a phosphatase that counteracts the ac...,"[PREX2, MAST2, PTK2, DLG1, TP53, AKT1, PIK3R1,...","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0}",Entry\tEntry Name\tProtein names\tGene Names\t...,"[f63d5ac5d778fd156d013f96d2998554, c223fa7c552...","[0.005102965049445629, -0.03068516217172146, 0...",\n1. PI3K - PTEN is a phosphatase that counte...,[PTEN is a phosphatase that counteracts the ac...
2,KISS1,[GPR54],"[14 - This is a cleavage product of KISS1, and...","[UMPS, LEP, GNRH1, KISS1R, TACR3, PROK2, TAC3,...","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0}",Entry\tEntry Name\tProtein names\tGene Names\t...,"[1566c63f2e619e2d64afed6fa486b5d3, a96164d6ca3...","[0.03324534371495247, -0.06768038123846054, 0....","\nBased on the provided context, I can predic...","[This is another name for the KiSS-1 receptor,..."
3,TP53,"[TP53BP1, TP53TG3, TP53TG5]","[TP53-binding protein 1, a known direct intera...","[HIF1A, CHEK2, HDAC1, HSP90AA1, MDM4, BCL2L1, ...","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0}",Entry\tEntry Name\tProtein names\tGene Names\t...,"[57fd6cfde1b94a6bc86ef9ef4628d8e5, 9697344e8c6...","[0.06858199089765549, -0.06567219644784927, 0....","TP53BP1 - TP53-binding protein 1, a known dir...","[TP53-binding protein 1, a known direct intera..."


In [52]:
df_documents_agg

,name,accession,synonyms,organism,id,description,go_terms,pathways,gene_sets,num_questions,question_ids
0,AAK1_HUMAN,Q2M2I8,AAK1 KIAA1048,Homo sapiens (Human),5e8e102f14c68d5bbb617f913a2d3b45,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,"membrane organization [GO:0061024], positive r...","R-HSA-8856825, R-HSA-8856828, hsa:22848","3D-structure, Acetylation, Alternative splicin...",0,[]
1,AAK1_MOUSE,Q3UHJ0,Aak1 Kiaa1048,Mus musculus (Mouse),eb411854dce15cdf4e2044ca6052ddc2,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,positive regulation of Notch signaling pathway...,"R-MMU-8856825, R-MMU-8856828, mmu:269774","3D-structure, Acetylation, Alternative splicin...",0,[]
2,AAK1_BOVIN,F1MH24,AAK1,Bos taurus (Bovine),658b7b833376e83107222dd24ea1de2b,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,"endocytosis [GO:0006897], positive regulation ...",,"Acetylation, ATP-binding, Cell membrane, Cell ...",1,[AAK1]
3,AAK1_RAT,P0C1X8,Aak1,Rattus norvegicus (Rat),60e9447efcc478b8bf0ebac372fe42cb,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,positive regulation of Notch signaling pathway...,"R-RNO-8856825, R-RNO-8856828","Acetylation, ATP-binding, Cell membrane, Cell ...",0,[]
4,AAK1_PIG,F1SPM8,AAK1,Sus scrofa (Pig),16802aabf322829fa025af97c1f1f4fb,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,"endocytosis [GO:0006897], positive regulation ...",,"Acetylation, ATP-binding, Cell membrane, Cell ...",1,[AAK1]
...,...,...,...,...,...,...,...,...,...,...,...
95,P53_MARMO,O36006,TP53,Marmota monax (Woodchuck),cabedd041bbafc3d8a1a0731d671106a,Cellular tumor antigen p53 (Tumor suppressor p53),"cellular senescence [GO:0090398], circadian be...",,"Acetylation, Activator, Apoptosis, Biological ...",0,[]
96,P53_PIG,Q9TUB2,TP53 P53,Sus scrofa (Pig),d29afdf50a9c2d217edc47ac7aad44f6,Cellular tumor antigen p53 (Tumor suppressor p53),"cellular senescence [GO:0090398], circadian be...","R-SSC-2559580, R-SSC-2559584, R-SSC-2559586, R...","Acetylation, Activator, Apoptosis, Biological ...",0,[]
97,P53_HORSE,P79892,TP53 P53,Equus caballus (Horse),b0bd57d8a4635e6dfab183ef056e8d4a,Cellular tumor antigen p53 (Tumor suppressor p53),"apoptotic process [GO:0006915], circadian beha...",,"Acetylation, Activator, Apoptosis, Biological ...",0,[]
98,T53G3_HUMAN,Q9ULZ0,TP53TG3 TP53TG3A; TP53TG3B; TP53TG3C; TP53TG3D...,Homo sapiens (Human),57fd6cfde1b94a6bc86ef9ef4628d8e5,TP53-target gene 3 protein (TP53-inducible gen...,"cytoplasm [GO:0005737], nucleus [GO:0005634]","hsa:102724101, hsa:102724127, hsa:24150, hsa:6...","Alternative splicing, Cytoplasm, Nucleus, Refe...",1,[TP53]


In [53]:
df

,protein,predictions,reasoning,ground_truth,metrics,contexts,source_documents,embedding,raw_answer,reasons,...,organism,id,description,go_terms,pathways,gene_sets,num_questions,question_ids,umap_x,umap_y
0,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,05be9863936dea49c55ce86cd5088cb6,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Homo sapiens (Human),05be9863936dea49c55ce86cd5088cb6,5'-AMP-activated protein kinase catalytic subu...,"autophagy [GO:0006914], CAMKK-AMPK signaling c...","R-HSA-1632852, R-HSA-380972, R-HSA-5628897, R-...","3D-structure, Alternative splicing, ATP-bindin...",1,[AAK1],7.568156,5.023273
1,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,658b7b833376e83107222dd24ea1de2b,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Bos taurus (Bovine),658b7b833376e83107222dd24ea1de2b,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,"endocytosis [GO:0006897], positive regulation ...",,"Acetylation, ATP-binding, Cell membrane, Cell ...",1,[AAK1],8.165907,4.762008
2,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,16802aabf322829fa025af97c1f1f4fb,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Sus scrofa (Pig),16802aabf322829fa025af97c1f1f4fb,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,"endocytosis [GO:0006897], positive regulation ...",,"Acetylation, ATP-binding, Cell membrane, Cell ...",1,[AAK1],7.671310,4.344192
3,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,3203c054eb57e0584333c40d87d317ec,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Homo sapiens (Human),3203c054eb57e0584333c40d87d317ec,E3 ubiquitin-protein ligase XIAP (EC 2.3.2.27)...,"copper ion homeostasis [GO:0055070], defense r...","R-HSA-111459, R-HSA-111463, R-HSA-111464, R-HS...","3D-structure, Apoptosis, Cytoplasm, Isopeptide...",1,[AAK1],8.328205,3.371215
4,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,725bfca67356f82b202e84faeee52ddd,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Sus scrofa (Pig),725bfca67356f82b202e84faeee52ddd,5'-AMP-activated protein kinase catalytic subu...,"autophagy [GO:0006914], cellular response to g...",,"ATP-binding, Autophagy, Biological rhythms, Ch...",1,[AAK1],8.139897,3.964528
5,PTEN,"[PI3K, AKT, TSC2, LKB1, PDK1, 14-3-3]",[PTEN is a phosphatase that counteracts the ac...,"[PREX2, MAST2, PTK2, DLG1, TP53, AKT1, PIK3R1,...","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0}",Entry\tEntry Name\tProtein names\tGene Names\t...,f63d5ac5d778fd156d013f96d2998554,"[0.005102965049445629, -0.03068516217172146, 0...",\n

In [61]:
df

,protein,predictions,reasoning,ground_truth,metrics,contexts,source_documents,embedding,raw_answer,reasons,...,organism,id,description,go_terms,pathways,gene_sets,num_questions,question_ids,umap_x,umap_y
0,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,05be9863936dea49c55ce86cd5088cb6,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Homo sapiens (Human),05be9863936dea49c55ce86cd5088cb6,5'-AMP-activated protein kinase catalytic subu...,"autophagy [GO:0006914], CAMKK-AMPK signaling c...","R-HSA-1632852, R-HSA-380972, R-HSA-5628897, R-...","3D-structure, Alternative splicing, ATP-bindin...",1,[AAK1],7.568156,5.023273
1,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,658b7b833376e83107222dd24ea1de2b,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Bos taurus (Bovine),658b7b833376e83107222dd24ea1de2b,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,"endocytosis [GO:0006897], positive regulation ...",,"Acetylation, ATP-binding, Cell membrane, Cell ...",1,[AAK1],8.165907,4.762008
2,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,16802aabf322829fa025af97c1f1f4fb,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Sus scrofa (Pig),16802aabf322829fa025af97c1f1f4fb,AP2-associated protein kinase 1 (EC 2.7.11.1) ...,"endocytosis [GO:0006897], positive regulation ...",,"Acetylation, ATP-binding, Cell membrane, Cell ...",1,[AAK1],7.671310,4.344192
3,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,3203c054eb57e0584333c40d87d317ec,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Homo sapiens (Human),3203c054eb57e0584333c40d87d317ec,E3 ubiquitin-protein ligase XIAP (EC 2.3.2.27)...,"copper ion homeostasis [GO:0055070], defense r...","R-HSA-111459, R-HSA-111463, R-HSA-111464, R-HS...","3D-structure, Apoptosis, Cytoplasm, Isopeptide...",1,[AAK1],8.328205,3.371215
4,AAK1,"[AP2M1, CLTC, NOTCH1, APP, TSG101]",[AAK1 is involved in clathrin-dependent endocy...,"[RALBP1, CLTB, AP2B1, EPS15, AP2M1, SGIP1, AP2...","{'precision': 0.2, 'recall': 1.0, 'f1': 0.3333...",Entry\tEntry Name\tProtein names\tGene Names\t...,725bfca67356f82b202e84faeee52ddd,"[0.03289015218615532, -0.026425722986459732, 0...","\nBased on the provided information, here are...",[AAK1 is known to interact with the AP-2 adapt...,...,Sus scrofa (Pig),725bfca67356f82b202e84faeee52ddd,5'-AMP-activated protein kinase catalytic subu...,"autophagy [GO:0006914], cellular response to g...",,"ATP-binding, Autophagy, Biological rhythms, Ch...",1,[AAK1],8.139897,3.964528
5,PTEN,"[PI3K, AKT, TSC2, LKB1, PDK1, 14-3-3]",[PTEN is a phosphatase that counteracts the ac...,"[PREX2, MAST2, PTK2, DLG1, TP53, AKT1, PIK3R1,...","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0}",Entry\tEntry Name\tProtein names\tGene Names\t...,f63d5ac5d778fd156d013f96d2998554,"[0.005102965049445629, -0.03068516217172146, 0...",\n

In [63]:
# import sys
# if "umap" in sys.modules:
#     del sys.modules["umap"]
from umap import UMAP 
import plotly.express as px

umap = UMAP(
    n_neighbors=20,
    min_dist=0.15,
    metric="cosine",
    random_state=42
)

embeddings = df["embedding"].values.tolist()
umap_all = umap.fit_transform(embeddings)

df["umap_x"] = umap_all[:, 0]
df["umap_y"] = umap_all[:, 1]

fig = px.scatter(
    df,
    x="umap_x",
    y="umap_y",
    color="protein", 
    hover_data={
        "name": True,
        "accession": True,
        "synonyms": True,
        "organism": True,
        "umap_x": False,
        "umap_y": False
    },
    title="Similarity map of protein context embeddings",
    width=800,
    height=600
)

fig.update_traces(marker=dict(size=10, opacity=0.7, line=dict(width=0.5, color="white")))
fig.show()

/Users/allison/anaconda3/envs/protein-env/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.

/Users/allison/anaconda3/envs/protein-env/lib/python3.10/site-packages/umap/umap_.py:2462: UserWarning:

n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1

